# 🔍 Explication de modèle - SHAP & Grad-CAM

Ce notebook charge un modèle entraîné et applique des techniques d'explicabilité.

In [ ]:
import sys
import os

# Se placer à la racine du projet pour les chemins relatifs
os.chdir('..')
sys.path.append('.')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from data_provider.data_loader import load_data_train_val_test
from models import DLinear, PatchTST, PatchMixer
from tools.gradcam_utils import grad_cam_timeseries

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Working directory: {os.getcwd()}")

sns.set_style('whitegrid')
np.random.seed(42)
torch.manual_seed(42)

## 1. Charger le modèle

In [ ]:
# Choisir le modèle à expliquer
MODEL_NAME = "PatchTST"  # Changer ici: DLinear, PatchTST, PatchMixer

# Charger la config correspondante
config_files = {
    'DLinear': 'configs/linearModel.json',
    'PatchTST': 'configs/patchTST.json',
    'PatchMixer': 'configs/patchMixer.json'
}

with open(config_files[MODEL_NAME], 'r') as f:
    config = json.load(f)

class Args:
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

args = Args(config)

print(f"Configuration chargée pour {MODEL_NAME}:")
print(f"  - Data: {args.data}")
print(f"  - Input length: {args.input_len}")
print(f"  - Prediction length: {args.pred_len}")

# Charger le modèle
model_dict = {'DLinear': DLinear, 'PatchTST': PatchTST, 'PatchMixer': PatchMixer}
model = model_dict[args.model].Model(args).to(device)

# Charger les poids
checkpoint_path = f"checkpoints/{args.model}_baseline.pth"
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"✓ Modèle chargé depuis {checkpoint_path}")
else:
    print(f"⚠ Pas de checkpoint trouvé: {checkpoint_path}")
    print(f"⚠ Entraînez d'abord avec 1_train_baselines.ipynb")

model.eval()
print(f"Paramètres: {sum(p.numel() for p in model.parameters()):,}")

## 2. Charger les données

In [ ]:
train_loader, val_loader, test_loader, scaler = load_data_train_val_test(args, args.data)

# Prendre un batch de test
test_batch = next(iter(test_loader))
batch_x, batch_y, batch_idx = test_batch
batch_x, batch_y = batch_x.to(device), batch_y.to(device)

print(f"Batch shape: {batch_x.shape}")
print(f"Features: level, PRELIQ_Q, T_Q, ETP_Q")

## 3. Grad-CAM - Importance temporelle

In [ ]:
# Calculer Grad-CAM pour 1 sample
sample_x = batch_x[:1]
sample_y = batch_y[:1]

# Prédiction
with torch.no_grad():
    pred = model(sample_x)

# Grad-CAM sur la prédiction du dernier timestep de 'level' (feature 0)
target_fn = lambda out: out[0, -1, 0]

try:
    gradcam_scores = grad_cam_timeseries(args, model, sample_x, target=target_fn, device=device)
    gradcam_scores = gradcam_scores[0]  # Premier sample
    print(f"✓ Grad-CAM calculé: shape {gradcam_scores.shape}")
except Exception as e:
    print(f"⚠ Grad-CAM échoué: {e}")
    print("Note: Grad-CAM fonctionne mieux avec PatchTST/PatchMixer")
    gradcam_scores = None

In [ ]:
if gradcam_scores is not None:
    # Visualiser Grad-CAM
    fig, ax = plt.subplots(figsize=(14, 4))
    
    # Série temporelle (level)
    ax.plot(sample_x[0, :, 0].cpu().numpy(), label='Input (level)', 
            marker='o', linewidth=2, color='blue', alpha=0.7)
    
    # Grad-CAM scores en rouge
    ax2 = ax.twinx()
    ax2.fill_between(range(len(gradcam_scores)), gradcam_scores, 
                      alpha=0.3, color='red', label='Importance Grad-CAM')
    ax2.set_ylabel('Grad-CAM Score', color='red')
    
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Niveau d\'eau (normalisé)', color='blue')
    ax.set_title(f'Grad-CAM - {args.model}\nImportance des timesteps pour prédire le niveau futur')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Créer le dossier figs/ s'il n'existe pas
    os.makedirs('figs', exist_ok=True)
    plt.savefig(f'figs/{args.model}_gradcam_explanation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Top-k timesteps
    top_k = 5
    top_indices = np.argsort(gradcam_scores)[-top_k:][::-1]
    print(f"\nTop {top_k} timesteps les plus importants:")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank}. Timestep {idx} (t-{args.input_len-idx}): score={gradcam_scores[idx]:.4f}")

## 4. Gradient x Input - Importance par feature

In [ ]:
def compute_gradient_importance(model, input_tensor):
    """
    Calcule l'importance via Gradient x Input.
    """
    model.eval()
    input_tensor.requires_grad = True
    
    # Forward
    output = model(input_tensor)
    
    # Target: dernière prédiction de 'level'
    target = output[0, -1, 0]
    
    # Backward
    target.backward()
    
    # Importance = |gradient × input|
    importance = (input_tensor.grad * input_tensor).abs()
    
    return importance[0].detach().cpu().numpy()

sample_input = batch_x[:1].clone()
importance_matrix = compute_gradient_importance(model, sample_input)

print(f"Importance shape: {importance_matrix.shape}")
print(f"  [timesteps={args.input_len}, features=4]")

In [ ]:
# Heatmap d'importance
feature_names = ['level', 'PRELIQ_Q', 'T_Q', 'ETP_Q']

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(importance_matrix.T, 
            cmap='YlOrRd', 
            yticklabels=feature_names,
            cbar_kws={'label': 'Importance'},
            ax=ax)
ax.set_xlabel('Timestep')
ax.set_ylabel('Feature')
ax.set_title(f'{args.model} - Importance par timestep et feature\n(Gradient × Input)')
plt.tight_layout()
plt.savefig(f'figs/{args.model}_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Importance agrégée par feature
feature_importance = importance_matrix.sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(feature_names, feature_importance, 
              color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
ax.set_ylabel('Importance totale')
ax.set_title(f'{args.model} - Importance globale des features')
ax.grid(True, alpha=0.3, axis='y')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nImportance relative:")
for name, score in zip(feature_names, feature_importance):
    pct = 100 * score / feature_importance.sum()
    print(f"  {name:12s}: {pct:5.1f}%")

## 5. SHAP (optionnel - si installé)

In [ ]:
try:
    import shap
    print("✓ SHAP installé")
    
    # Wrapper pour SHAP
    def model_predict(x):
        x_tensor = torch.FloatTensor(x).to(device)
        with torch.no_grad():
            pred = model(x_tensor)
        return pred[:, -1, 0].cpu().numpy()  # Dernier timestep, feature 0
    
    # Créer explainer
    background = batch_x[:50].cpu().numpy()
    explainer = shap.KernelExplainer(model_predict, background)
    
    # Calculer SHAP values
    test_sample = batch_x[:5].cpu().numpy()
    shap_values = explainer.shap_values(test_sample, nsamples=100)
    
    print(f"SHAP values shape: {shap_values.shape}")
    
    # Visualiser
    shap.summary_plot(shap_values.reshape(-1, args.input_len * 4), 
                      test_sample.reshape(-1, args.input_len * 4),
                      show=False)
    plt.tight_layout()
    plt.savefig(f'figs/{args.model}_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    
except ImportError:
    print("⚠ SHAP non installé")
    print("  Pour l'installer: pip install shap")
except Exception as e:
    print(f"⚠ SHAP échoué: {e}")
    print("  Note: SHAP peut être lent sur modèles complexes")

## 6. Analyse multi-samples

In [ ]:
# Analyser plusieurs samples pour statistiques robustes
n_samples = min(20, len(test_loader.dataset))
all_importance = []

print(f"Analyse de {n_samples} samples...")
for i, (bx, by, _) in enumerate(test_loader):
    if i * args.batch_size >= n_samples:
        break
    
    for j in range(min(args.batch_size, n_samples - i * args.batch_size)):
        sample = bx[j:j+1].to(device).clone()
        imp = compute_gradient_importance(model, sample)
        all_importance.append(imp)

all_importance = np.array(all_importance)
mean_importance = all_importance.mean(axis=0)
std_importance = all_importance.std(axis=0)

print(f"✓ {len(all_importance)} samples analysés")

In [ ]:
# Visualiser importance moyenne ± std pour chaque feature
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (ax, name) in enumerate(zip(axes, feature_names)):
    mean_vals = mean_importance[:, i]
    std_vals = std_importance[:, i]
    
    ax.plot(mean_vals, marker='o', linewidth=2, label='Moyenne')
    ax.fill_between(range(len(mean_vals)), 
                     mean_vals - std_vals, 
                     mean_vals + std_vals, 
                     alpha=0.3, label='±1 std')
    
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Importance')
    ax.set_title(f'{name} - Importance temporelle')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'{args.model} - Analyse multi-samples (n={len(all_importance)})', fontsize=14)
plt.tight_layout()
plt.savefig(f'figs/{args.model}_multisamples_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Sauvegarde des résultats

In [ ]:
# Sauvegarder les résultats d'explication
results = {
    'model': args.model,
    'feature_names': feature_names,
    'mean_importance': mean_importance.tolist(),
    'std_importance': std_importance.tolist(),
    'feature_importance_total': mean_importance.sum(axis=0).tolist(),
    'n_samples_analyzed': len(all_importance)
}

if gradcam_scores is not None:
    results['gradcam_scores'] = gradcam_scores.tolist()

os.makedirs('results/explainability', exist_ok=True)
output_path = f'results/explainability/{args.model}_explanation.json'

with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✓ Résultats sauvegardés: {output_path}")
print(f"✓ Figures sauvegardées dans: figs/")

## Conclusion

**Insights clés :**

1. **Grad-CAM** identifie les timesteps importants pour la prédiction
2. **Gradient × Input** montre l'importance de chaque feature à chaque timestep
3. **SHAP** (si installé) fournit des explications model-agnostic

Ces méthodes permettent de:
- Valider que le modèle capture les bonnes dynamiques hydrologiques
- Identifier les variables/périodes critiques
- Débugger les prédictions aberrantes